# 🏦 Pandas pour auditeurs — Solutions Niveau 2 : Moyen

**Contexte** : Revue du portefeuille de prêts immobiliers.

> ⚠️ Ce fichier contient les **solutions**. Essayez d'abord avec `exercice_moyen.ipynb` !

## 0. Génération des données

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
np.random.seed(2025)

n = 350

agences = ['Agence Nord', 'Agence Sud', 'Agence Est', 'Agence Ouest', 'Agence Centre']
types_pret = ['Immobilier résidentiel', 'Immobilier professionnel', 'Investissement locatif']
statuts = ['Payé', 'Retard léger', 'Retard grave', 'Impayé']

statut_paiement = np.random.choice(statuts, n, p=[0.68, 0.15, 0.10, 0.07])
montants_echeance = np.round(np.random.uniform(500, 4500, n), 2)

taux_paiement = np.where(
    statut_paiement == 'Payé', 1.0,
    np.where(statut_paiement == 'Retard léger',
             np.random.uniform(0.0, 1.0, n),
             np.where(statut_paiement == 'Retard grave',
                      np.random.uniform(0.0, 0.5, n),
                      0.0))
)
montants_payes = np.round(montants_echeance * taux_paiement, 2)

jours_retard = np.where(
    statut_paiement == 'Payé', 0,
    np.where(statut_paiement == 'Retard léger', np.random.randint(1, 30, n),
    np.where(statut_paiement == 'Retard grave', np.random.randint(30, 90, n),
             np.random.randint(90, 365, n)))
)

prets = pd.DataFrame({
    'pret_id':          [f'PR{str(i).zfill(5)}' for i in range(1, n + 1)],
    'client_id':        np.random.randint(50000, 51000, n),
    'agence':           np.random.choice(agences, n),
    'type_pret':        np.random.choice(types_pret, n, p=[0.55, 0.25, 0.20]),
    'taux_interet':     np.round(np.random.uniform(1.5, 4.5, n), 2),
    'date_echeance':    pd.to_datetime('2024-01-01') + pd.to_timedelta(
                            np.random.randint(0, 180, n), unit='D'),
    'montant_echeance': montants_echeance,
    'montant_paye':     montants_payes,
    'statut_paiement':  statut_paiement,
    'nb_jours_retard':  jours_retard,
})

prets.loc[np.random.choice(prets.index, 10, replace=False), 'taux_interet'] = np.nan
prets.loc[np.random.choice(prets.index, 5,  replace=False), 'montant_paye'] = np.nan

dups = prets.sample(4, random_state=99).copy()
dups['pret_id'] = [f'DUP{i}' for i in range(4)]
prets = pd.concat([prets, dups], ignore_index=True)
prets = prets.sample(frac=1, random_state=3).reset_index(drop=True)

print('Jeu de données prêts prêt :', prets.shape[0], 'échéances,', prets.shape[1], 'colonnes')
prets.head()

---
## Exercice 1 — Filtrage multi-conditions

In [ ]:
# 1. Retard grave ou Impayé
problematiques = prets[prets['statut_paiement'].isin(['Retard grave', 'Impayé'])]
print('Retards graves + Impayés :', len(problematiques))
problematiques.head()

In [ ]:
# 2. Impayés avec montant > 3 000
prets[(prets['statut_paiement'] == 'Impayé') & (prets['montant_echeance'] > 3000)][
    ['pret_id', 'agence', 'type_pret', 'montant_echeance', 'nb_jours_retard']
]

In [ ]:
# 3. Retard entre 30 et 90 jours
retard_moyen = prets[prets['nb_jours_retard'].between(30, 90)]
print('Échéances avec retard 30-90 jours :', len(retard_moyen))
retard_moyen[['pret_id', 'agence', 'statut_paiement', 'nb_jours_retard', 'montant_echeance']].head()

In [ ]:
# 4. Prêts pro/locatif NON payés
types_cibles = ['Immobilier professionnel', 'Investissement locatif']
non_payes_pro = prets[
    prets['type_pret'].isin(types_cibles)
    & ~(prets['statut_paiement'] == 'Payé')
]
print('Prêts pro/locatif non payés :', len(non_payes_pro))
non_payes_pro[['pret_id', 'agence', 'type_pret', 'statut_paiement', 'montant_echeance']].head()

---
## Exercice 2 — Colonnes calculées

In [ ]:
# 1. Montant impayé
prets['montant_impaye'] = (prets['montant_echeance'] - prets['montant_paye'].fillna(0)).round(2)
prets[['montant_echeance', 'montant_paye', 'montant_impaye']].head(8)

In [ ]:
# 2. Retard critique (booléen)
prets['retard_critique'] = prets['nb_jours_retard'] >= 60
print('Retards critiques (>= 60j) :', prets['retard_critique'].sum())

In [ ]:
# 3. Catégorie de retard avec np.select
conditions = [
    prets['nb_jours_retard'] == 0,
    prets['nb_jours_retard'].between(1, 59),
]
choix = ['Aucun', 'Léger']
prets['categorie_retard'] = np.select(conditions, choix, default='Critique')
prets['categorie_retard'].value_counts()

---
## Exercice 3 — Synthèses avec `groupby`

In [ ]:
# 1. Synthèse par agence
synthese_agence = prets.groupby('agence').agg(
    total_du=('montant_echeance', 'sum'),
    total_paye=('montant_paye', 'sum'),
    total_impaye=('montant_impaye', 'sum'),
).round(2)
synthese_agence.sort_values('total_impaye', ascending=False)

In [ ]:
# 2. Synthèse par type de prêt
prets.groupby('type_pret').agg(
    nb_echeances=('pret_id', 'count'),
    retard_moyen_jours=('nb_jours_retard', 'mean'),
    impaye_total=('montant_impaye', 'sum'),
).round(2)

In [ ]:
# 3. Nombre d'échéances par agence et statut
prets.groupby(['agence', 'statut_paiement'])['pret_id'].count().unstack(fill_value=0)

---
## Exercice 4 — Tableau croisé (`pivot_table`)

In [ ]:
# 1. Pivot : montant impayé par agence × type de prêt
pd.pivot_table(
    prets,
    index='agence',
    columns='type_pret',
    values='montant_impaye',
    aggfunc='sum',
    fill_value=0,
).round(2)

In [ ]:
# 2. Pivot : nombre d'échéances par agence × statut
pd.pivot_table(
    prets,
    index='agence',
    columns='statut_paiement',
    values='pret_id',
    aggfunc='count',
    fill_value=0,
)

---
## Exercice 5 — Qualité des données

In [ ]:
# 1. Valeurs manquantes par colonne
prets.isna().sum()

In [ ]:
# 2. Doublons métier
cles_metier = ['client_id', 'date_echeance', 'montant_echeance']
doublons = prets[prets.duplicated(subset=cles_metier, keep=False)]
print('Lignes impliquées dans un doublon :', len(doublons))
doublons.sort_values(cles_metier)[['pret_id', 'client_id', 'date_echeance', 'montant_echeance']].head(10)

In [ ]:
# 3. DataFrame nettoyé
prets_clean = (
    prets
    .drop_duplicates(subset=cles_metier, keep='first')
    .copy()
)
prets_clean['montant_paye'] = prets_clean['montant_paye'].fillna(0)
print('Lignes avant nettoyage :', len(prets))
print('Lignes après nettoyage :', len(prets_clean))

---
## Exercice 6 — Analyse de synthèse : agences à risque

In [ ]:
# Travail sur le DataFrame nettoyé
total_par_agence = prets_clean.groupby('agence').agg(
    nb_total=('pret_id', 'count'),
    impaye_total=('montant_impaye', 'sum'),
)

# Nombre de retards graves + impayés
defauts = prets_clean[prets_clean['statut_paiement'].isin(['Retard grave', 'Impayé'])]
nb_defauts = defauts.groupby('agence')['pret_id'].count().rename('nb_defauts')

# Fusion et calcul du taux
risque = total_par_agence.join(nb_defauts, how='left').fillna(0)
risque['taux_defaillance_pct'] = (risque['nb_defauts'] / risque['nb_total'] * 100).round(1)
risque['impaye_total'] = risque['impaye_total'].round(2)

risque.sort_values('taux_defaillance_pct', ascending=False)